# Time Series Forecasting with Sine Wave Data

This notebook demonstrates a basic time series preprocessing pipeline:
1. Generate synthetic sine wave data
2. Split into train/test sets
3. Scale features using MinMaxScaler

### Generate Data & Visualize

Create 501 evenly spaced points over [0, 50] and compute `sin(x)` to produce a clean periodic signal.

In [0]:
import numpy as np
import pandas as pd
%matplotlib inline
import matplotlib.pyplot as plt

# Generate 501 evenly spaced points in [0, 50] and compute sin(x)
x = np.linspace(0, 50, 501)
y = np.sin(x)

# Quick visual sanity check of the raw periodic signal
plt.plot(x, y)
plt.xlabel('x')
plt.ylabel('sin(x)')
plt.title('Synthetic Sine Wave (501 samples)')
plt.show()

### Train / Test Split

Reserve the last 10% of the series as the test set. This preserves temporal order (no shuffle), which is critical for time series.

In [0]:
# Build a DataFrame indexed by x-values with the sine signal as target column
df = pd.DataFrame(data=y, index=x, columns=['Sine'])

# Reserve the last 10% as the test set (temporal split, no shuffling)
test_percent = 0.1
test_point = np.round(len(df) * test_percent)
test_ind = int(len(df) - test_point)

train = df.iloc[:test_ind]   # first 451 samples
test = df.iloc[test_ind:]    # last 50 samples

print(f"Train: {len(train)} samples | Test: {len(test)} samples")

### Feature Scaling

Apply `MinMaxScaler` fitted on the training set only, then transform both train and test to the [0, 1] range to avoid data leakage.

In [0]:
from sklearn.preprocessing import MinMaxScaler

# Fit scaler on training data only to prevent data leakage
scaler = MinMaxScaler(feature_range=(0, 1))
train_scaled = scaler.fit_transform(train)   # fit + transform on train
test_scaled = scaler.transform(test)          # transform only on test

print(f"Scaled range — train: [{train_scaled.min():.2f}, {train_scaled.max():.2f}] | test: [{test_scaled.min():.2f}, {test_scaled.max():.2f}]")

### Model Building & Training

`TimeseriesGenerator` creates sliding windows of `length` consecutive time steps as input features, with the next value as the target — automating the supervised-learning transformation for sequential data.

We use a **SimpleRNN** with 50 units followed by a single Dense output, trained with Adam/MSE and early stopping on validation loss.

### Recursive Multi-Step Forecasting

Generate predictions for the full test horizon by feeding each prediction back as input for the next step (autoregressive rollout). Finally, inverse-transform scaled predictions back to the original sine amplitude and plot against ground truth.

In [0]:
from tensorflow.keras.preprocessing.sequence import TimeseriesGenerator

# Sliding window length: use 50 prior time steps to predict the next value
length = 50
batch_size = 1

# TimeseriesGenerator produces (X, y) pairs where X is a window of `length`
# consecutive scaled values and y is the immediately following value
generator = TimeseriesGenerator(train_scaled, train_scaled, length=length, batch_size=batch_size)

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, SimpleRNN, GRU

n_features = 1  # univariate time series

# Build a simple RNN: 50 hidden units -> 1 Dense output
model = Sequential()
model.add(SimpleRNN(50, input_shape=(length, n_features)))  # recurrent layer
model.add(Dense(1))  # single output: next predicted value
model.compile(optimizer='adam', loss='mse')  # regression task

model.summary()

from tensorflow.keras.callbacks import EarlyStopping

# Stop training if validation loss doesn't improve for 2 consecutive epochs
early_stop = EarlyStopping(monitor='val_loss', patience=2)

# Train the model (using train generator as both train and validation for demo)
model.fit(generator, epochs=10, validation_data=generator, callbacks=[early_stop])

# Plot training vs validation loss curves
loss_df = pd.DataFrame(model.history.history)
loss_df.plot()

In [0]:
# --- Recursive (autoregressive) multi-step forecasting ---

# Seed the prediction loop with the last `length` training values
first_eval_batch = train_scaled[-length:]
first_eval_batch = first_eval_batch.reshape((1, length, n_features))
model.predict(first_eval_batch)  # warm-up call

test_predictions = []

# Re-initialize the sliding window for the actual forecast loop
first_eval_batch = train_scaled[-length:]
current_batch = first_eval_batch.reshape((1, length, n_features))

# Roll forward: predict one step, append prediction, slide window by 1
for i in range(len(test)):
    current_pred = model.predict(current_batch)[0]  # predict next value
    test_predictions.append(current_pred)
    # Drop oldest time step, append new prediction at the end
    current_batch = np.append(current_batch[:, 1:, :], [[current_pred]], axis=1)

# Inverse-transform predictions back to original sine scale
rnn_predictions = scaler.inverse_transform(test_predictions)
# Use test_scaled (not test) for inverse transform — test is already in original scale
test_actual = scaler.inverse_transform(test_scaled)

# Store for comparison in the next cell
print(f"SimpleRNN forecast complete: {len(rnn_predictions)} steps")

In [0]:
# --- LSTM model: same task, different architecture ---
early_stop = EarlyStopping(monitor='val_loss', patience=2)
length_lstm = 49
batch_size = 1
n_features = 1

# Re-create generators using scaled data
generator_lstm = TimeseriesGenerator(train_scaled, train_scaled, length=length_lstm, batch_size=batch_size)
validation_generator = TimeseriesGenerator(test_scaled, test_scaled, length=length_lstm, batch_size=batch_size)

# Build LSTM model: 50 units -> Dense(1)
model_lstm = Sequential()
model_lstm.add(LSTM(50, input_shape=(length_lstm, n_features)))
model_lstm.add(Dense(1))
model_lstm.compile(optimizer='adam', loss='mse')

model_lstm.fit(generator_lstm, epochs=20, validation_data=validation_generator, callbacks=[early_stop])

# --- LSTM recursive forecasting ---
lstm_predictions = []
first_eval_batch = train_scaled[-length_lstm:]
current_batch = first_eval_batch.reshape((1, length_lstm, n_features))

for i in range(len(test_actual)):
    current_pred = model_lstm.predict(current_batch)[0]
    lstm_predictions.append(current_pred)
    current_batch = np.append(current_batch[:, 1:, :], [[current_pred]], axis=1)

lstm_predictions = scaler.inverse_transform(lstm_predictions)

# --- Combined plot: original data + both NN predictions ---
fig, ax = plt.subplots(figsize=(14, 6))

# Full original sine wave (train + test) using actual x-axis indices
ax.plot(train.index, train['Sine'].values, color='blue', label='Training Data')
ax.plot(test.index, test_actual[:, 0], color='green', label='Test (Actual)', linewidth=2)

# Overlay both model predictions on the test region (aligned to test x-axis)
ax.plot(test.index, rnn_predictions[:, 0], color='red', linestyle='--', label='SimpleRNN Predictions', linewidth=2)
ax.plot(test.index, lstm_predictions[:, 0], color='orange', linestyle='-.', label='LSTM Predictions', linewidth=2)

ax.set_xlabel('Time Step')
ax.set_ylabel('Sine Value')
ax.set_title('Sine Wave Forecasting: SimpleRNN vs LSTM')
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()